# Deep Learning HW 3 Part 2
Author: Daniel Bond

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd

from ner_utils import (WordTokenizer, BPETokenizer, load_entities, merge_touching, parse_markers, label_names,
                       spans_to_tags, decode_locations, comp_metric)

seed = 67
random.seed(seed)
np.random.seed(seed)

DATA_DIR = Path("data")
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)
RU_ROWS = 300_000
RU_VALID_EVERY = 10

uk_df = pd.read_csv(DATA_DIR / "uk_geo_dataset.csv")
ru_df = pd.read_csv(DATA_DIR / "ru_geo_dataset.csv", nrows=RU_ROWS)
sample_df = pd.read_csv(DATA_DIR / "labeling_sample.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

uk_df["split"] = np.where(uk_df["is_valid"] == 1, "valid", "train")
ru_df["split"] = np.where(ru_df["doc_id"] % RU_VALID_EVERY == 0, "valid", "train")
sample_df["locations"] = sample_df["locations"].map(eval)
print(f"uk_geo: {uk_df.shape}, ru_geo: {ru_df.shape}, labeling_sample: {sample_df.shape}, test: {test_df.shape}")

uk_geo: (1010000, 6), ru_geo: (300000, 7), labeling_sample: (26, 3), test: (477, 3)


## Validation

The splits are fixed by rules (`is_valid` for `uk_geo`, `doc_id % 10` for `ru_geo`), so every model in Part 3 is trained and validated on exactly the same data.


In [2]:
rows = {}
for name, df in [("uk_geo", uk_df), ("ru_geo", ru_df)]:
    for split in ["train", "valid"]:
        part = df[df["split"] == split]
        rows[f"{name} {split}"] = {"texts": len(part), "LOC spans": part["loc_markers"].str.count(r"\(").sum(),
                                   "docs": part["doc_id"].nunique() if "doc_id" in part else np.nan}
rows["labeling_sample"] = {"texts": len(sample_df), "LOC spans": sample_df["locations"].str.len().sum(), "docs": np.nan}
display(pd.DataFrame(rows).T)

uk_train_texts = set(uk_df.loc[uk_df["split"] == "train", "text"])
ru_train_docs = set(ru_df.loc[ru_df["split"] == "train", "doc_id"])
print(f"uk valid texts also in uk train: {uk_df.loc[uk_df['split'] == 'valid', 'text'].isin(uk_train_texts).sum()}")
print(f"ru valid docs also in ru train: {len(set(ru_df.loc[ru_df['split'] == 'valid', 'doc_id']) & ru_train_docs)}")
print(f"duplicated texts inside uk train: {uk_df.loc[uk_df['split'] == 'train', 'text'].duplicated().sum()}")
all_train = uk_train_texts | set(ru_df["text"])
print(f"labeling_sample texts in training data: {sample_df['text'].isin(all_train).sum()}, test: {test_df['text'].isin(all_train).sum()}")

,texts,LOC spans,docs
uk_geo train,1000000.0,341810.0,NaN
uk_geo valid,10000.0,3304.0,NaN
ru_geo train,269928.0,117084.0,23169.0
ru_geo valid,30072.0,12844.0,2575.0
labeling_sample,26.0,32.0,NaN


uk valid texts also in uk train: 52
ru valid docs also in ru train: 0
duplicated texts inside uk train: 2402


labeling_sample texts in training data: 0, test: 0



- Validation has 3.3k LOC spans in `uk_geo` and 12.8k in `ru_geo` - enough for stable scores. No `ru_geo` document is split between train and validation.
- 52 `uk_geo` valid sentences also appear in train, and train has 2.4k duplicates - short common sentences, too few to inflate the score, so I keep them.
- Neither the sample nor the test texts appear in the training data.

#### Expected correlation with the LB
I expect a positive but weak correlation between silver validation and the LB:
- short news sentences vs long Telegram posts with emojis and line breaks
- different location fields (countries vs streets)
- `*_geo`'s labels are another model's output, with its errors
- the sample and the public LB are small, noise is probable

#### Adversarial validation
News and test texts can be told apart by eye, so the question is how strong it is. Runs in Part 3.


### From spans to tags and back

In [3]:
TYPES = ["LOC", "ORG", "PER"]
LABELS = label_names(TYPES)
uk_entities = load_entities(uk_df, TYPES)
ru_entities = load_entities(ru_df, TYPES)

raw_spans = [parse_markers(m) for m in uk_df["loc_markers"]]
n_merged = sum(len(r) - len(e["LOC"]) for r, e in zip(raw_spans, uk_entities))
print(f"uk_geo LOC spans merged (touching parts of one name): {n_merged}")
print(LABELS)

example = "в м. Івано-Франківську на вул. Шевченка"
word_tok = WordTokenizer.train([example])
ids, offsets = word_tok.encode(example)
tags = spans_to_tags(offsets, {"LOC": [(5, 10), (10, 11), (11, 22), (26, 39)]}, len(example))
pd.DataFrame({"token": [example[a:b] for a, b in offsets], "tag": tags}).T

uk_geo LOC spans merged (touching parts of one name): 3490
['O', 'B-LOC', 'I-LOC', 'B-ORG', 'I-ORG', 'B-PER', 'I-PER']


,0,1,2,3,4,5,6,7
token,в,м,.,Івано-Франківську,на,вул,.,Шевченка
tag,O,O,O,B-LOC,O,B-LOC,I-LOC,I-LOC


### Tokenizers

In [4]:
tokenizer_texts = pd.concat([uk_df.loc[uk_df["split"] == "train", "text"], ru_df.loc[ru_df["split"] == "train", "text"]])
tokenizers = {
    "word": WordTokenizer.train(tokenizer_texts, max_size=100_000, min_freq=2),
    "bpe": BPETokenizer.train(tokenizer_texts, vocab_size=32_000),
}
for name, tok in tokenizers.items():
    tok.save(ARTIFACTS / f"{name}_tokenizer.json")
    print(f"{name}: vocab {tok.vocab_size}")

bpe_example = "Мешканці Здолбунівської вулиці та Троєщини"
for name, tok in tokenizers.items():
    ids, offsets = tok.encode(bpe_example)
    print(f"{name}:", [bpe_example[a:b] if i != tok.unk_id else "<unk>" for i, (a, b) in zip(ids, offsets)])

word: vocab 100003
bpe: vocab 32000
word: ['Мешканці', '<unk>', 'вулиці', 'та', 'Троєщини']
bpe: ['Мешканці', 'З', 'дол', 'бу', 'нів', 'ської', 'вулиці', 'та', 'Троє', 'щини']


In [5]:
def oracle(tok, texts, entities):
    """Predictions of a model that outputs the true tags for every token."""
    loc_labels = label_names(["LOC"])
    preds = []
    for text, ents in zip(texts, entities):
        _, offsets = tok.encode(text)
        tags = spans_to_tags(offsets, {"LOC": ents["LOC"]}, len(text))
        probs = np.eye(len(loc_labels))[[loc_labels.index(t) for t in tags]]
        preds.append(decode_locations(text, offsets, probs, loc_labels))
    return preds


def tokenizer_stats(tok, texts):
    ids = [tok.encode(text)[0] for text in texts]
    return np.mean([len(i) for i in ids]), np.mean([t == tok.unk_id for i in ids for t in i]) * 100


uk_valid = uk_df["split"] == "valid"
valid_texts = uk_df.loc[uk_valid, "text"].tolist()
valid_entities = [e for e, v in zip(uk_entities, uk_valid) if v]
valid_gold_raw = [[t[a:b] for a, b in parse_markers(m)] for t, m in zip(valid_texts, uk_df.loc[uk_valid, "loc_markers"])]
valid_gold = [[t[a:b] for a, b in e["LOC"]] for t, e in zip(valid_texts, valid_entities)]
sample_entities = [{"LOC": merge_touching(parse_markers(m))} for m in sample_df["loc_markers"]]
sample_locations = [l for locs in sample_df["locations"] for l in locs]

rows = {}
for name, tok in tokenizers.items():
    uk_len, uk_unk = tokenizer_stats(tok, valid_texts)
    test_len, test_unk = tokenizer_stats(tok, test_df["text"])
    rows[name] = {
        "vocab": tok.vocab_size,
        "tokens / text uk valid": uk_len,
        "tokens / text test": test_len,
        "unk % uk valid": uk_unk,
        "unk % test": test_unk,
        "unk % sample locations": tokenizer_stats(tok, sample_locations)[1],
        "oracle F1 uk valid (raw labels)": comp_metric(valid_gold_raw, oracle(tok, valid_texts, valid_entities)),
        "oracle F1 uk valid (merged)": comp_metric(valid_gold, oracle(tok, valid_texts, valid_entities)),
        "oracle F1 labeling_sample": comp_metric(sample_df["locations"].tolist(), oracle(tok, sample_df["text"], sample_entities)),
    }
pd.DataFrame(rows).round(4)

,word,bpe
vocab,100003.0000,32000.0000
tokens / text uk valid,17.6621,22.0017
tokens / text test,72.8197,95.1677
unk % uk valid,6.2082,0.0005
unk % test,11.0436,1.4429
unk % sample locations,27.6596,0.0000
oracle F1 uk valid (raw labels),0.9651,0.9651
oracle F1 uk valid (merged),0.9770,0.9770
oracle F1 labeling_sample,1.0000,1.0000



Both tokenizers are trained on the training splits only:
- word-level: regex words, punctuation and line breaks
- BPE: 32k subwords learned inside the same words

Results:
- 11% of test tokens and 28% of the words in the sample locations are `<unk>` with word-level. On test the unknowns are also emoji, Telegram boilerplate and abbreviations.
- BPE has no unknowns in the sample locations, its 1.4% on test are emoji and symbols that never occur in news.
- The price of BPE is longer sequences and subword pieces that mean nothing alone.